# 🌿 DOFBOT Planting Hole Detector - YOLOv5n Training on GPU

This notebook fine-tunes **YOLOv5n** on your multi-angle planting hole dataset with free NVIDIA GPU acceleration in ~2-3 minutes.

### Steps:
1. Ensure GPU is enabled (`Runtime` -> `Change runtime type` -> select `T4 GPU`).
2. Upload `dataset_colab.zip` to Colab.
3. Run all cells below.
4. Download `best.pt` and place it into `hole_detector/models/` on your computer.

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install Ultralytics
!pip install -q ultralytics

In [ ]:
# Step 3: Unzip your dataset
!unzip -q /content/dataset_colab.zip -d /content/

# Check dataset contents
!ls -lh /content/dataset/images/train | head -n 5

In [ ]:
# Step 4: Create data.yaml configuration
yaml_content = """
path: /content/dataset
train: images/train
val: images/val

names:
  0: empty
  1: filled
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("data.yaml configured successfully.")

In [ ]:
# Step 5: Train YOLOv5n on GPU (50 epochs in ~2-3 minutes!)
from ultralytics import YOLO

model = YOLO('yolov5nu.pt')

results = model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,            # Use GPU
    project='colab_runs',
    name='planting_hole_yolov5n',
    degrees=15.0,        # rotation augmentation
    perspective=0.0005,  # perspective augmentation for angles
    shear=5.0,           # shear augmentation
    fliplr=0.5,
    scale=0.5
)

In [ ]:
# Step 6: Export to ONNX (for Jetson Nano / Pi)
exported_onnx = model.export(format='onnx', imgsz=640, simplify=True)
print(f"ONNX model ready at: {exported_onnx}")

In [ ]:
# Step 7: Download trained models to your computer
from google.colab import files
import os

best_pt = 'colab_runs/planting_hole_yolov5n/weights/best.pt'
if os.path.exists(best_pt):
    print("Downloading best.pt...")
    files.download(best_pt)
    
best_onnx = 'colab_runs/planting_hole_yolov5n/weights/best.onnx'
if os.path.exists(best_onnx):
    print("Downloading best.onnx...")
    files.download(best_onnx)